In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -q unsloth transformers datasets accelerate trl peft bitsandbytes

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

In [ ]:
max_seq_length = 2048
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
    dtype=torch.float16,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing=True,
)

In [ ]:
dataset = load_dataset("databricks/databricks-dolly-15k")

train_data = dataset["train"].select(range(600))

In [ ]:
SYSTEM_PROMPT = (
    "You are a friendly, concise, and professional Technical Support Expert. "
    "Always respond in English."
)

PROMPT_TEMPLATE = (
    "<|begin_of_text|>"
    "<|start_header_id|>system<|end_header_id|>\n"
    "{system_prompt}<|eot_id|>"
    "<|start_header_id|>user<|end_header_id|>\n"
    "{user_instruction}<|eot_id|>"
    "<|start_header_id|>assistant<|end_header_id|>\n"
    "{assistant_answer}<|eot_id|>"
)

In [ ]:
def format_example(example):
    return {
        "text": PROMPT_TEMPLATE.format(
            system_prompt=SYSTEM_PROMPT,
            user_instruction=example["instruction"].strip(),
            assistant_answer=example["response"].strip(),
        )
    }

train_data = train_data.map(format_example)

In [ ]:
training_config = SFTConfig(
    max_seq_length=max_seq_length,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    fp16=True,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    dataset_text_field="text",
    args=training_config,
)

In [ ]:
trainer.train()

In [ ]:
FastLanguageModel.for_inference(model)

In [ ]:
def generate_reply(instruction):
    prompt = PROMPT_TEMPLATE.format(
        system_prompt=SYSTEM_PROMPT,
        user_instruction=instruction,
        assistant_answer=""
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        use_cache=False,
    )

    output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return output.split("assistant")[-1].strip()

print(generate_reply("My laptop shows a blue screen error, what should I do?"))

In [ ]:
from huggingface_hub import login
login(token="HF_TOKEN")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "unsloth/Llama-3.2-3B-Instruct"
ADAPTER_PATH = "fine_tuned_slm"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

model = model.merge_and_unload()

In [ ]:
model = model.to("cpu")

model.save_pretrained("merged_model", safe_serialization=True)
tokenizer.save_pretrained("merged_model")

In [ ]:
HF_REPO = "aijadugar/ft_slm_model"

model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

In [ ]:
import requests

API_URL = "https://router.huggingface.co/hf-inference/models/aijadugar/ft_slm_model"
headers = {"Authorization": "Bearer HF_TOKEN"}

def query(payload):
    response = requests.post(API_URL, headers=headers, json=payload)
    return response.json()

print(query({"inputs": "my mouse right click is not working? what should I do!"}))